In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [26]:
df = pd.read_csv('../data/train.txt', sep=';',header=None , names=['text','emotion'])

In [27]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [28]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [29]:
unique_emotions = df['emotion'].unique()

In [30]:
emotion_numners = {}
i = 0
for emo in unique_emotions:
    emotion_numners[emo] = i
    i += 1

In [31]:
emotion_numners

{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}

In [32]:
df['emotion'] = df['emotion'].map(emotion_numners)

In [33]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [34]:
df['text'] = df['text'].apply(lambda x : x.lower())

In [35]:
import string


In [36]:
def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))


In [37]:
df['text']= df['text'].apply(remove_punc)

In [38]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [39]:
def remove_numbers(txt):
  new = ""
  for i in txt:
    if not i.isdigit():
      new = new+i
  return new

df['text'] = df['text'].apply(remove_numbers)

In [40]:

def remove_emoji(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_emoji)

In [41]:
import nltk

In [42]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [43]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [44]:
stop_words = set(stopwords.words('english'))

In [45]:
def remove(txt):
  word_tokens = word_tokenize(txt)
  cleaned = []
  for i in word_tokens:
    if i not in stop_words:
      cleaned.append(i)
  return " ".join(cleaned)

In [46]:
df['text'] = df['text'].apply(remove)

In [47]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [48]:
from sklearn.model_selection import train_test_split

In [49]:
X = df['text']
y = df['emotion']

In [50]:
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [51]:
from sklearn.feature_extraction.text import TfidfVectorizer , CountVectorizer

In [52]:
bow = CountVectorizer()
tfidf = TfidfVectorizer()

In [53]:
X_train_bow = bow.fit_transform(X_train)
X_train_tfidf = tfidf.fit_transform(X_train)

In [54]:
X_test_bow = bow.transform(X_test)
X_test_tfidf = tfidf.transform(X_test)

In [55]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [56]:
nb1 = MultinomialNB()
nb2 = MultinomialNB()

In [57]:
nb_bow = nb1.fit(X_train_bow,y_train)
nb_tfidf = nb2.fit(X_train_tfidf,y_train)

In [58]:
pred_bow = nb_bow.predict(X_test_bow)
pred_tfidf = nb_tfidf.predict(X_test_tfidf)

In [59]:
acc_bow = accuracy_score(y_test,pred_bow)
acc_tfidf = accuracy_score(y_test,pred_tfidf)


In [60]:
print(acc_bow)
print(acc_tfidf)

0.7678125
0.6609375


In [61]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0], shape=(3200,))

In [62]:
from sklearn.linear_model import LogisticRegression

In [63]:
lg1 = LogisticRegression(max_iter=1000)
lg2 = LogisticRegression(max_iter=1000)

In [64]:
lg_bow = lg1.fit(X_train_bow,y_train)
lg_tfidf = lg2.fit(X_train_tfidf,y_train)

In [65]:
lg_pred_bow = lg_bow.predict(X_test_bow)
lg_pred_tfidf = lg_tfidf.predict(X_test_tfidf)

In [66]:
acc_log_bow = accuracy_score(y_test,lg_pred_bow)
acc_log_tfidf = accuracy_score(y_test,lg_pred_tfidf)

In [67]:
print(acc_log_bow)
print(acc_log_tfidf)

0.88875
0.8615625


In [68]:
import joblib

# pick whichever combo had the best accuracy - example uses Logistic Regression + TF-IDF
joblib.dump(lg_tfidf, 'model.pkl')
joblib.dump(tfidf, 'vectorizer.pkl')
joblib.dump(emotion_numners, 'label_map.pkl')

print("Saved model.pkl, vectorizer.pkl, label_map.pkl")

Saved model.pkl, vectorizer.pkl, label_map.pkl
